# Tensorflow VAE - it has only EDA and PPG , containing 2 stage model running

In [4]:
import os
os.chdir("/notebooks")

In [5]:
from lib.install import prepare_env
_ = prepare_env()


Installed versions:
tensorflow: 2.18.0
tensorflow-probability: 0.25.0
tf-keras: 2.18.0
transformers: 4.46.3
protobuf: 4.25.8
datasets: 5.0.0
evaluate: 0.4.6
scikit-learn: 1.9.0
imbalanced-learn: 0.14.2
wordcloud: 1.9.6
umap-learn: 0.5.12

Environment packages OK.


2026-06-21 13:29:15.506893: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782048555.639046      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782048555.681691      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-21 13:29:16.053355: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



TensorFlow check:
TensorFlow: 2.18.0
Built with CUDA: True
GPUs: []


2026-06-21 13:29:25.365992: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


#### 1. Import

In [11]:
import pandas as pd
import numpy as np
import random
from pathlib import Path
import tensorflow as tf
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE, RandomOverSampler
from sklearn.neighbors import NearestNeighbors
from lib.logger import Logger

import tensorflow_probability as tfp
from lib.vae_r.vae import VAE
tfkl = tf.keras.layers
tfd = tfp.distributions
tfk = tf.keras

#### 2. Config

In [12]:
hidden_layers = (100, 50)
latent_dim    = hidden_layers[1]     # equivalent to 2nd hd
batch_size    = 200 
epochs        = 200
seed_values   = [42, 43, 111]
lr            = 1e-3                 # adam learning rate
TASK          = os.getenv("TASK", "VALENCE")
MODALITY      = os.getenv("MODALITY", "EDA")
EXPERIMENT    = os.getenv("EXPERIMENT","SIMPLE")
MODEL_NAME    = "VAE"


FSCORE_CALCULATION = 'binary'

DATA_CONFIG = {
    "eda_path": "EEVR/Data_files/eda.csv",
    "ppg_path": "EEVR/Data_files/ppg.csv",
    "text_path": "EEVR/Data_files/text_data_labels.csv",
}

hidden_layers = (100, 50) - it controls sixe of the encoder and decoder network, input features → hd 100 → latent mean/log var 50, if i choose (10,20) model would work but it will not learn
complex patterns of the data. #(1000,500) strong overfitting

Batch size = (16, 32) can improve generalization, more updates per epoch
but more noise in gradients after loso and balancing total rows=340 or 350, so in 1 epoch
model would have 2 batches, model will update weights twice

epoch(20,50) = model may underfit, reconstructions and classification may not converge
if epochs are 500, 1000 = Overfitting to training participants, test performance may 
not improve.

Seed = vae training depends on the random weight initialization
and random shuffling, one seed might give good or bad results so 3 is 
good number. adding more seeds would increase the reliability but model will
take more time in training and test

learning rate - if it is slow model wont learn enough in 200 epochs
if it is very large loss will keep fluctuating



#### 3.1 Utility Functions

In [13]:
# ============================================================
# Utility: CSV read
# ============================================================
def csv_read(path):
    df = pd.read_csv(path)
    return df
        
logger = Logger(MODEL_NAME,MODALITY,TASK,EXPERIMENT)
#can say output dir for writing the models, logger.print would save the results
# in output folder 

# ============================================================
# Utility: seed everything
# ============================================================
def set_seed(seed=42):# default seed value
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed) # different part of codes need different
    np.random.seed(seed) # libraries for randomness
    tf.random.set_seed(seed)
    return seed 

# ============================================================
# Utility: Feature Analysis
# ============================================================
def feature_analysis(df, relevant_features, identifiers):
    df_features = df[relevant_features]
    arousals = df["arousal_category"].tolist()
    num_high_arousal = arousals.count(1)
    num_low_arousal = arousals.count(0)
    #print(df.head(5))
    logger.print(
        f"The Number of entries of High Arousal: {num_high_arousal} "
        f"& Low Arousal: {num_low_arousal}"
    )
    valence = df["valence_category"].tolist()
    num_high_valence = valence.count(1)
    num_low_valence = valence.count(0)

    logger.print(
        f"The Number of entries of High Valence: {num_high_valence} "
        f"& Low Valence: {num_low_valence}"
    )
    taskwiselabel_list = df["taskwiselabel"].tolist()
    num_zeros = taskwiselabel_list.count(0)
    num_ones = taskwiselabel_list.count(1)

    logger.print(
        f"The Number of entries of Positive Task: {num_ones} "
        f"& Negative Task: {num_zeros}"
    )
    pi = df["Participant ID"]
    vi = df["Video ID"]
    df_required = df[relevant_features + identifiers]
    return df_required, pi, vi


# ============================================================
# Utility: metrics
# ============================================================
def compute_metrics(y_true, y_pred, n_classes): # y_true = true label, y_pre=predicted
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    accuracy = accuracy_score(y_true, y_pred)
    if FSCORE_CALCULATION.lower() == "weighted" or n_classes > 2:
        f1score = f1_score(y_true,y_pred,average="weighted",zero_division=0)
    else:
        f1score = f1_score(y_true,y_pred,average="binary",zero_division=0)
    return {"acc": accuracy,"f1": f1score}


# ============================================================
# Utility: balance data SIMPLE EEVR
# ============================================================
def balance_train_data_simple(X_train, y_train):
    """
    Duplicate the minority class once and concatenate it back.
    No shuffling is applied.
    """

    X_train = X_train.reset_index(drop=True)                         # reset feature index
    y_train = y_train.reset_index(drop=True)                         # reset label index

    counts = y_train.value_counts()                                  # class counts
    minority_class = counts.idxmin()                                 # minority class label

    minority_idx = y_train[y_train == minority_class].index           # minority row indices

    X_minority = X_train.loc[minority_idx]                            # minority feature rows
    y_minority = y_train.loc[minority_idx]                            # minority labels

    X_balanced = pd.concat(
        [X_train, X_minority],
        axis=0,
        ignore_index=True,
    )                                                                # add minority rows at the end

    y_balanced = pd.concat(
        [y_train, y_minority],
        axis=0,
        ignore_index=True,
    )                                                                # add minority labels at the end

    return X_balanced, y_balanced


# ============================================================
# Utility: balance data SMOTE
# ============================================================
def balance_train_data_smote(X_train, y_train, random_state=42):
    """
    Balance X_train and y_train using SMOTE.
    If the minority class has only one sample, RandomOverSampler is used instead.
    """

    # Reset index to avoid alignment issues
    X_train = X_train.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

    # Check class counts
    counts = y_train.value_counts()
    minority_count = counts.min()

    # Use RandomOverSampler if SMOTE cannot be applied
    if minority_count <= 1:
        sampler = RandomOverSampler(random_state=random_state)
    else:
        sampler = SMOTE(
            random_state=random_state,
            k_neighbors=min(5, minority_count - 1)
        )

    X_balanced, y_balanced = sampler.fit_resample(X_train, y_train)

    # Convert back to pandas format
    X_balanced = pd.DataFrame(X_balanced, columns=X_train.columns)
    y_balanced = pd.Series(y_balanced, name=y_train.name)

    return X_balanced, y_balanced

#### 3.2 Utility: Tensorflow model

In [14]:
# ============================================================
# Encoder - recognition model, probabilistic encoder
# ============================================================
class Encoder(tf.keras.Model): # custom tensorflow model
    def __init__(self, hidden_layers, name="encoder"): # why we are giving input dimension, keras automatically reads input.
        super().__init__(name=name)
        h1, h2 = hidden_layers
        self.enc_h = tfkl.Dense(h1, activation="relu") #100 , .dense(h1) means it converts any input to h1 i.e. 100
        self.enc_mu = tfkl.Dense(h2) #50 we have already written self.mu = tfkl.dense(h2) now it just needs h
        self.enc_log_var = tfkl.Dense(h2) #50
        self.latent_dim = h2

    def call(self, x, training=False): #x shape = (batch_size, input_dim)
        h = self.enc_h(x) #h shape = (batch_size, 100(h1))
        z_mean = self.enc_mu(h) # z_mean shape = (batch size, 50)
        z_log_var = self.enc_log_var(h)
        z_std = tf.exp(0.5 * z_log_var) + 1e-6 # log variance to std
        qz_x = tfd.MultivariateNormalDiag(loc=z_mean, scale_diag=z_std) #q(z | x) = Normal(mean, variance)
        return qz_x, x # qz_x is distribution


# ============================================================
# Decoder
# ============================================================
class Decoder(tf.keras.Model):
    def __init__(self, output_dim, hidden_layers, name="decoder"):
        super().__init__(name=name) # model initialization.
        h1, h2 = hidden_layers
        self.dec_h = tfkl.Dense(h1, activation="relu")
        self.dec_mu = tfkl.Dense(output_dim) # mean shape (200, 15)
        self.dec_log_var = tfkl.Dense(output_dim) # logvar shape (200, 15) therefore z shape is also (200,50)

    def call(self, z, training=False):
        h = self.dec_h(z) # it is going from 50 to 100
        x_recon_mean = self.dec_mu(h) # x_hat is usually x_recon_mean
        x_recon_log_var = self.dec_log_var(h)
        x_recon_std = tf.exp(0.5 * x_recon_log_var) + 1e-6 # logvar -> var ->std
        px_z = tfd.MultivariateNormalDiag(loc=x_recon_mean, scale_diag=x_recon_std) #p(x | z) = Normal(reconstructed_mean, reconstructed_variance)
        return px_z


# ============================================================
# Classifier
# ============================================================
# This classifier takes the trained encoder.
# It can either freeze encoder or fine-tune encoder.
class Classifier(tf.keras.Model):
    def __init__(self, encoder, train_encoder=False, name="classifier"): # initialising binary classifier
        super().__init__(name=name) # initialize tf model
        self.encoder = encoder
        self.train_encoder = train_encoder
        self.out = tf.keras.layers.Dense(1) # BCE with logits needs one output logit

    @property
    def params(self): # trainable parameters
        if self.train_encoder:
            return self.encoder.trainable_variables + self.out.trainable_variables
        else:
            return self.out.trainable_variables

    def call(self, x, training=False):
        qz_x, _ = self.encoder(x, training=training)
        z = qz_x.mean() # Classifier predicts class using mean latent vector, not sampled z.
        logits = self.out(z) # one raw logit, shape = (batch_size, 1)
        return logits # z → Dense(1) → logit

    # any raw value that is later used to convert to prob called logit
    # one logit: positive value means class 1, negative value means class 0

    def compute_loss(self, x, y):
        logits = self.call(x, training=True)

        y = tf.cast(y, tf.float32)
        y = tf.reshape(y, (-1, 1)) # BCE expects same shape as logits: (batch_size, 1)

        bce = tf.nn.sigmoid_cross_entropy_with_logits(labels=y, logits=logits) # BCE loss with logits
        self.bce_loss = tf.reduce_mean(bce) # avg BCE loss over the batch
        self.loss = self.bce_loss
        return self.loss

    @tf.function
    def train(self, x, y, optimizer):
        with tf.GradientTape() as tape: # records all operations needed for gradient calculation
            self.compute_loss(x, y)

        gradients = tape.gradient(self.loss, self.params)
        optimizer.apply_gradients(zip(gradients, self.params))
        return self.loss

    # ============================================================
    # Inference functions
    # ============================================================
    def predict_logits(self, x):
        qz_x, _ = self.encoder(x, training=False)
        z = qz_x.mean()
        logits = self.out(z)
        return tf.squeeze(logits, axis=-1).numpy()

    def predict_probs(self, x):
        logits = self.predict_logits(x)
        probs = tf.sigmoid(logits)
        return probs.numpy()

    def predict_class(self, x):
        probs = self.predict_probs(x)
        y_pred = (probs >= 0.5).astype("int32")
        return y_pred

#### 4. Read Data

In [15]:
identifiers = [
    "Participant ID",
    "Video ID",
    "arousal_category",
    "valence_category",
    "taskwiselabel"
]


if MODALITY == 'EDA':
    features = [
        "ku_eda", "sk_eda", "dynrange", "slope", "variance",
        "entropy", "insc", "fd_mean", "max_scr", "min_scr",
        "nSCR", "meanAmpSCR", "meanRespSCR", "sumAmpSCR", "sumRespSCR",
    ]
    df = csv_read(DATA_CONFIG["eda_path"])
    df_required, df_pi, df_vi = feature_analysis(
        df,
        features,
        identifiers,
    )
elif MODALITY == 'PPG':
    features = ["BPM", "IBI", "PPG_Rate_Mean", "HRV_MedianNN", "HRV_Prc20NN", "HRV_MinNN", "HRV_HTI", "HRV_TINN", "HRV_LF",
                "HRV_VHF", "HRV_LFn", "HRV_HFn", "HRV_LnHF", "HRV_SD1SD2", "HRV_CVI", "HRV_PSS", "HRV_PAS", "HRV_PI",
                "HRV_C1d", "HRV_C1a", "HRV_DFA_alpha1", "HRV_MFDFA_alpha1_Width", "HRV_MFDFA_alpha1_Peak", "HRV_MFDFA_alpha1_Mean",
                "HRV_MFDFA_alpha1_Max", "HRV_MFDFA_alpha1_Delta", "HRV_MFDFA_alpha1_Asymmetry", "HRV_ApEn", "HRV_ShanEn",
                "HRV_FuzzyEn", "HRV_MSEn", "HRV_CMSEn", "HRV_RCMSEn", "HRV_CD", "HRV_HFD", "HRV_KFD", "HRV_LZC"]

    df = csv_read(DATA_CONFIG["ppg_path"])
    df_required, df_pi, df_vi = feature_analysis(
        df,
        features,
        identifiers,
    )
else:
    raise Error(f'Invalid Modality {MODALITY}')


if TASK == 'AROUSAL':
    label_column = 'arousal_category'
    n_class=2
elif TASK == 'VALENCE':
    label_column = 'valence_category'
    n_class=2
elif TASK == 'STIMULUS-LABEL':
    label_column = 'taskwiselabel'
    n_class=2
else:
    raise Error(f'Invalid task {TASK}')

The Number of entries of High Arousal: 114 & Low Arousal: 219
The Number of entries of High Valence: 202 & Low Valence: 131
The Number of entries of Positive Task: 148 & Negative Task: 185


In [16]:
X = df_required[features]
y = df_required[label_column]
participant_id = df_pi
video_id       = df_vi

### 5. Run LOSO Validation


In [ ]:
performance_rows = []

for seed in seed_values:
    set_seed(seed)                                                      # repeat experiment for each seed
    loso = LeaveOneGroupOut()                                           # LOSO split by participant

    for fold_no, (train_idx, test_idx) in enumerate(loso.split(X, y, groups=participant_id), start=1):
        test_pid = participant_id.iloc[test_idx].iloc[0]                # held-out participant

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]           # train/test features
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]           # train/test labels

        sample_id = participant_id.iloc[test_idx].astype(str).values + "_" + video_id.iloc[test_idx].astype(str).values  # participant_video id
        
        if EXPERIMENT=="SMOTE":
            X_train,y_train = balance_train_data_smote(X_train,y_train) # balance train data using SMOTE
        elif EXPERIMENT=="SIMPLE":
            X_train,y_train = balance_train_data_simple(X_train,y_train) # balance train data by duplication
        
        X_train = np.asarray(X_train).astype("float32")                 # TensorFlow expects float array
        y_train = np.asarray(y_train).astype("int32")                   # labels as integer array
        X_test  = np.asarray(X_test).astype("float32")                  # test features
        y_test  = np.asarray(y_test).astype("int32")                    # test labels

        dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(len(X_train), seed=seed).batch(batch_size) # TF batches
        
        # ============================================================
        # Stage 1: Train VAE only
        # ============================================================
        encoder = Encoder(hidden_layers)                                # probabilistic encoder q(z|x)
        decoder = Decoder(X_train.shape[1], hidden_layers)              # probabilistic decoder p(x|z)
        vae = VAE(tfd.MultivariateNormalDiag, encoder, decoder)         # VAE model

        vae_optimizer = tf.keras.optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.999, epsilon=1e-8) # Adam optimizer

        for epoch in range(epochs):
            for x_batch, y_batch in dataset:
                vae.train(x_batch, vae_optimizer, L=1)                  # train VAE with one latent sample

        # ============================================================
        # Stage 2: Train classifier using trained encoder
        # ============================================================
        classifier = Classifier(encoder=encoder, train_encoder=True)    # classifier fine-tunes encoder

        classifier_optimizer = tf.keras.optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.999, epsilon=1e-8) # Adam optimizer

        for epoch in range(epochs):
            for x_batch, y_batch in dataset:
                classifier.train(x_batch, y_batch, classifier_optimizer) # train classifier

        pred_proba = classifier.predict_probs(X_test)                   # P(class 1)
        y_pred = classifier.predict_class(X_test)                       # class prediction using threshold 0.5

        logger.add_predictions(
            seed=seed,
            fold=fold_no,
            pid=test_pid,
            sample_id=sample_id,
            y_true=y_test,
            pred_proba=pred_proba,
        )                                                              # save gt, pred_proba, pred_class_0.5

        metrics = compute_metrics(y_test, y_pred, n_class)              # compute accuracy and F1

        performance_rows.append({"seed": seed, "pid": test_pid, "acc": metrics["acc"], "f1": metrics["f1"]}) # store fold result

        perf_df = pd.DataFrame(performance_rows)                        # all fold results so far
        seed_mean_df = perf_df.groupby("seed")[["acc", "f1"]].mean().reset_index() # mean over participants per seed
        global_mean = seed_mean_df[["acc", "f1"]].mean()                # mean over seeds
        global_std = seed_mean_df[["acc", "f1"]].std()                  # std over seeds

        logger.print(f"Running Mean | Seed={seed} | pid={test_pid} | Acc={global_mean['acc']:.4f} | F1={global_mean['f1']:.4f}")

summary_df = pd.DataFrame([{"seed": "Mean ± Std", "acc": f"{global_mean['acc']:.4f} ± {global_std['acc']:.4f}", "f1": f"{global_mean['f1']:.4f} ± {global_std['f1']:.4f}"}])
seed_summary_df = pd.concat([seed_mean_df, summary_df], axis=0, ignore_index=True)

logger.print("Seed-wise mean results:")
logger.print(seed_summary_df)
logger.save_npz()                                                       # save sample_id, gt, pred_proba, pred_class_0.5, threshold, seed, fold, pid


I0000 00:00:1781851898.471279 1183035 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15657 MB memory:  -> device: 0, name: Quadro P5000, pci bus id: 0000:00:05.0, compute capability: 6.1
2026-06-19 06:51:45.971302: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-06-19 06:51:45.990543: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-06-19 06:51:46.024443: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-06-19 06:51:46.094675: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-06-19 06:51:46.221590: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-06-19

Running Mean | Seed=42 | pid=1 | Acc=0.6667 | F1=0.4000
